<a href="https://colab.research.google.com/github/rakshitE/MachineLearning_261_VI/blob/main/ml_5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import math
from collections import Counter

# ---------- Entropy ----------
def entropy(labels):
    total = len(labels)
    counts = Counter(labels)
    ent = 0
    for count in counts.values():
        p = count / total
        ent -= p * math.log2(p)
    return ent


# ---------- Information Gain ----------
def information_gain(data, labels, feature_index):
    total_entropy = entropy(labels)
    total = len(labels)

    # Split by feature values
    feature_values = {}
    for i in range(total):
        value = data[i][feature_index]
        if value not in feature_values:
            feature_values[value] = []
        feature_values[value].append(labels[i])

    weighted_entropy = 0
    for subset in feature_values.values():
        weighted_entropy += (len(subset) / total) * entropy(subset)

    return total_entropy - weighted_entropy


# ---------- ID3 Algorithm ----------
def id3(data, labels, feature_names):
    # If all labels same → return leaf
    if len(set(labels)) == 1:
        return labels[0]

    # If no features left → return majority class
    if len(feature_names) == 0:
        return Counter(labels).most_common(1)[0][0]

    # Choose best feature
    gains = [information_gain(data, labels, i) for i in range(len(feature_names))]
    best_index = gains.index(max(gains))
    best_feature = feature_names[best_index]

    tree = {best_feature: {}}

    # Get unique feature values
    feature_values = set([row[best_index] for row in data])

    for value in feature_values:
        subset_data = []
        subset_labels = []

        for i in range(len(data)):
            if data[i][best_index] == value:
                reduced_row = data[i][:best_index] + data[i][best_index+1:]
                subset_data.append(reduced_row)
                subset_labels.append(labels[i])

        if not subset_data:
            tree[best_feature][value] = Counter(labels).most_common(1)[0][0]
        else:
            new_features = feature_names[:best_index] + feature_names[best_index+1:]
            tree[best_feature][value] = id3(subset_data, subset_labels, new_features)

    return tree



In [ ]:
data1 = [
    ['Sunny', 'Hot', 'High', 'Weak'],
    ['Sunny', 'Hot', 'High', 'Strong'],
    ['Overcast', 'Hot', 'High', 'Weak'],
    ['Rain', 'Mild', 'High', 'Weak'],
    ['Rain', 'Cool', 'Normal', 'Weak'],
    ['Rain', 'Cool', 'Normal', 'Strong'],
    ['Overcast', 'Cool', 'Normal', 'Strong'],
    ['Sunny', 'Mild', 'High', 'Weak'],
    ['Sunny', 'Cool', 'Normal', 'Weak'],
    ['Rain', 'Mild', 'Normal', 'Weak'],
    ['Sunny', 'Mild', 'Normal', 'Strong'],
    ['Overcast', 'Mild', 'High', 'Strong'],
    ['Overcast', 'Hot', 'Normal', 'Weak'],
    ['Rain', 'Mild', 'High', 'Strong']
]

labels1 = [
    'No','No','Yes','Yes','Yes','No','Yes',
    'No','Yes','Yes','Yes','Yes','Yes','No'
]

features1 = ['Outlook', 'Temperature', 'Humidity', 'Wind']

tree1 = id3(data1, labels1, features1)
print("Play Tennis Tree:")
print(tree1)

Play Tennis Tree:
{'Outlook': {'Rain': {'Wind': {'Weak': 'Yes', 'Strong': 'No'}}, 'Sunny': {'Humidity': {'Normal': 'Yes', 'High': 'No'}}, 'Overcast': 'Yes'}}


In [ ]:
data2 = [
    ['Young','Myope','No','Reduced'],
    ['Young','Myope','No','Normal'],
    ['Young','Myope','Yes','Reduced'],
    ['Young','Myope','Yes','Normal'],
    ['Pre','Hyper','No','Reduced'],
    ['Pre','Hyper','No','Normal'],
    ['Pre','Hyper','Yes','Reduced'],
    ['Pre','Hyper','Yes','Normal'],
    ['Old','Myope','No','Reduced'],
    ['Old','Myope','No','Normal'],
    ['Old','Hyper','Yes','Reduced'],
    ['Old','Hyper','Yes','Normal']
]

labels2 = [
    'None','Soft','None','Hard',
    'None','Soft','None','Hard',
    'None','Soft','None','Hard'
]

features2 = ['Age','Prescription','Astigmatic','TearRate']

tree2 = id3(data2, labels2, features2)
print("\nContact Lenses Tree:")
print(tree2)


Contact Lenses Tree:
{'TearRate': {'Normal': {'Astigmatic': {'Yes': 'Hard', 'No': 'Soft'}}, 'Reduced': 'None'}}


In [ ]:
import numpy as np

# -----------------------------
# Node structure
# -----------------------------
class Node:
    def __init__(self, feature=None, threshold=None, left=None, right=None, value=None):
        self.feature = feature      # index of feature
        self.threshold = threshold  # split value
        self.left = left
        self.right = right
        self.value = value          # for leaf nodes


# -----------------------------
# CART Regressor
# -----------------------------
class CARTRegressor:
    def __init__(self, max_depth=None, min_samples_split=2, min_samples_leaf=1):
        self.max_depth = max_depth
        self.min_samples_split = min_samples_split
        self.min_samples_leaf = min_samples_leaf
        self.root = None

    # -------------------------
    # Fit
    # -------------------------
    def fit(self, X, y):
        self.root = self._grow_tree(X, y, depth=0)

    # -------------------------
    # Grow Tree
    # -------------------------
    def _grow_tree(self, X, y, depth):
        n_samples, n_features = X.shape

        # Stopping conditions
        if (self.max_depth is not None and depth >= self.max_depth) or \
           n_samples < self.min_samples_split or \
           len(set(y)) == 1:
            return Node(value=np.mean(y))

        best_feature, best_threshold = self._best_split(X, y)

        if best_feature is None:
            return Node(value=np.mean(y))

        # Split
        left_idx = X[:, best_feature] <= best_threshold
        right_idx = X[:, best_feature] > best_threshold

        left = self._grow_tree(X[left_idx], y[left_idx], depth + 1)
        right = self._grow_tree(X[right_idx], y[right_idx], depth + 1)

        return Node(best_feature, best_threshold, left, right)

    # -------------------------
    # Find Best Split (MSE)
    # -------------------------
    def _best_split(self, X, y):
        n_samples, n_features = X.shape
        best_mse = float("inf")
        best_feature = None
        best_threshold = None

        for feature in range(n_features):
            thresholds = np.unique(X[:, feature])

            for threshold in thresholds:
                left_idx = X[:, feature] <= threshold
                right_idx = X[:, feature] > threshold

                if sum(left_idx) < self.min_samples_leaf or \
                   sum(right_idx) < self.min_samples_leaf:
                    continue

                mse = self._mse(y[left_idx], y[right_idx])

                if mse < best_mse:
                    best_mse = mse
                    best_feature = feature
                    best_threshold = threshold

        return best_feature, best_threshold

    # -------------------------
    # MSE calculation
    # -------------------------
    def _mse(self, left_y, right_y):
        total = len(left_y) + len(right_y)

        left_mse = np.var(left_y) * len(left_y)
        right_mse = np.var(right_y) * len(right_y)

        return (left_mse + right_mse) / total

    # -------------------------
    # Predict
    # -------------------------
    def predict(self, X):
        return np.array([self._predict_sample(x, self.root) for x in X])

    def _predict_sample(self, x, node):
        if node.value is not None:
            return node.value

        if x[node.feature] <= node.threshold:
            return self._predict_sample(x, node.left)
        else:
            return self._predict_sample(x, node.right)

In [ ]:
# Simple regression dataset
X = np.array([
    [1], [2], [3], [4], [5], [6], [7], [8]
])

y = np.array([2, 3, 4, 5, 8, 9, 10, 12])

model = CARTRegressor(max_depth=3)
model.fit(X, y)

predictions = model.predict(X)

print("Predictions:", predictions)

Predictions: [ 2.   3.   4.   5.   8.   9.5  9.5 12. ]


In [ ]:
import numpy as np
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Load dataset
data = fetch_california_housing()
X = data.data
y = data.target

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Train model
model = CARTRegressor(max_depth=10, min_samples_split=5)
model.fit(X_train, y_train)

# Predictions
y_pred = model.predict(X_test)

mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, y_pred)

print("MAE :", mae)
print("MSE :", mse)
print("RMSE:", rmse)
print("R2  :", r2)

MAE : 0.4314818670738957
MSE : 0.4085420308020176
RMSE: 0.6391729271504055
R2  : 0.6882331870539786
